@inproceedings{wilie2020indonlu,
  title={IndoNLU: Benchmark and Resources for Evaluating Indonesian Natural Language Understanding},
  author={Bryan Wilie and Karissa Vincentio and Genta Indra Winata and Samuel Cahyawijaya and X. Li and Zhi Yuan Lim and S. Soleman and R. Mahendra and Pascale Fung and Syafri Bahar and A. Purwarianti},
  booktitle={Proceedings of the 1st Conference of the Asia-Pacific Chapter of the Association for Computational Linguistics and the 10th International Joint Conference on Natural Language Processing},
  year={2020}
}

In [1]:
%pip install -U -q pip setuptools wheel setuptools-scm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.2/108.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 7.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -q -U \
    transformers==4.44.2 \
    datasets==2.21.0 \
    accelerate==0.34.2 \
    seqeval==1.2.2 \
    optimum \
    optimum-onnx \
    onnx==1.16.2 \
    onnxruntime==1.19.2 \
    onnxscript

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip uninstall -y diffusers

Found existing installation: diffusers 0.37.1
Uninstalling diffusers-0.37.1:
  Successfully uninstalled diffusers-0.37.1
Note: you may need to restart the kernel to use updated packages.


In [4]:
import glob
import time
from collections import Counter
from pathlib import Path

import numpy as np
import onnx
import onnxruntime as ort
from datasets import load_dataset
from onnxruntime.quantization import QuantType, quantize_dynamic
from seqeval.metrics import classification_report, f1_score
from transformers import (
    AutoModelForTokenClassification,
    BertTokenizerFast,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

from optimum.exporters.onnx import main_export
from optimum.onnxruntime import ORTModelForTokenClassification, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig
from onnxruntime.quantization.shape_inference import quant_pre_process

In [5]:
CANDIDATES = [
    ("indonlp/indonlu", "nerp"),
    ("indonlp/indonlu", "nergrit"),
    ("SEACrowd/indonlu_nergrit", None),
    ("SEACrowd/indonlu_nerp", None),
]

ds = None
for repo, cfg in CANDIDATES:
    try:
        ds = load_dataset(repo, cfg, trust_remote_code=True) if cfg else \
             load_dataset(repo, trust_remote_code=True)
        print(f"LOADED: {repo} config={cfg}")
        break
    except Exception as e:
        print(f"  failed {repo}/{cfg}: {str(e)[:90]}")

assert ds is not None, "None loaded — check HF Hub for current config names"
print()
print(ds)

Generating train split:   0%|          | 0/6720 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/840 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/840 [00:00<?, ? examples/s]

LOADED: indonlp/indonlu config=nerp

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 6720
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 840
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 840
    })
})


In [6]:
split = "train" if "train" in ds else list(ds.keys())[0]
print("columns:", ds[split].column_names)
print()
ex = ds[split][0]
for k, v in ex.items():
    print(f"{k}: {str(v)[:120]}")

columns: ['tokens', 'ner_tags']

tokens: ['kepala', 'dinas', 'tata', 'kota', 'manado', 'amos', 'kenda', 'menyatakan', 'tidak', 'tahu', '-', 'menahu', 'soal', 'pe
ner_tags: [9, 9, 9, 9, 2, 7, 0, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9]


In [7]:
TOKEN_COL = "tokens" if "tokens" in ds[split].column_names else ds[split].column_names[0]
TAG_COL   = [c for c in ds[split].column_names if c != TOKEN_COL][0]
print(f"tokens -> {TOKEN_COL!r}   tags -> {TAG_COL!r}")

feat = ds[split].features[TAG_COL]
try:
    LABELS = feat.feature.names         
except AttributeError:
    LABELS = sorted({t for row in ds[split][TAG_COL] for t in row})

LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
print(f"\n{len(LABELS)} labels:", LABELS)

tokens -> 'tokens'   tags -> 'ner_tags'

11 labels: ['I-PPL', 'B-EVT', 'B-PLC', 'I-IND', 'B-IND', 'B-FNB', 'I-EVT', 'B-PPL', 'I-PLC', 'O', 'I-FNB']


In [8]:
MODEL_NAME = "indobenchmark/indobert-base-p2"
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

print(type(tokenizer))
print(tokenizer.is_fast)

def tokenize_and_align(batch):
    enc = tokenizer(batch[TOKEN_COL], is_split_into_words=True,
                    truncation=True, max_length=128)
    all_labels = []
    for i, tags in enumerate(batch[TAG_COL]):
        word_ids = enc.word_ids(batch_index=i)
        prev, labs = None, []
        for wid in word_ids:
            if wid is None:
                labs.append(-100)                 
            elif wid != prev:
                t = tags[wid]
                labs.append(t if isinstance(t, int) else LABEL2ID[t])
            else:
                labs.append(-100)                 
            prev = wid
        all_labels.append(labs)
    enc["labels"] = all_labels
    return enc

tokenized = ds.map(tokenize_and_align, batched=True,
                   remove_columns=ds[split].column_names)
print(tokenized)

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

<class 'transformers.models.bert.tokenization_bert_fast.BertTokenizerFast'>
True


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/6720 [00:00<?, ? examples/s]

Map:   0%|          | 0/840 [00:00<?, ? examples/s]

Map:   0%|          | 0/840 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 6720
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 840
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 840
    })
})


In [9]:
row = tokenized[split][0]
toks = tokenizer.convert_ids_to_tokens(row["input_ids"])
for t, l in list(zip(toks, row["labels"]))[:25]:
    print(f"{t:<18} {ID2LABEL[l] if l != -100 else '(ignored)'}")

[CLS]              (ignored)
kepala             O
dinas              O
tata               O
kota               O
manado             B-PLC
am                 B-PPL
##os               (ignored)
kend               I-PPL
##a                (ignored)
menyatakan         O
tidak              O
tahu               O
-                  O
mena               O
##hu               (ignored)
soal               O
pencabutan         O
bali               O
##ho               (ignored)
.                  O
ia                 O
enggan             O
berkomentar        O
banyak             O


In [10]:
def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)
    tp = [[ID2LABEL[a] for a, b in zip(pr, lb) if b != -100] for pr, lb in zip(preds, labels)]
    tl = [[ID2LABEL[b] for a, b in zip(pr, lb) if b != -100] for pr, lb in zip(preds, labels)]
    return {"f1": f1_score(tl, tp)}

EVAL_SPLIT = "validation" if "validation" in tokenized else ("test" if "test" in tokenized else split)

def build(epochs, out):
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID)
    args = TrainingArguments(output_dir=out, num_train_epochs=epochs,
        per_device_train_batch_size=16, per_device_eval_batch_size=32,
        learning_rate=5e-5, eval_strategy="epoch", save_strategy="no",
        logging_steps=25, report_to="none")
    return Trainer(model=model, args=args,
        train_dataset=tokenized[split], eval_dataset=tokenized[EVAL_SPLIT],
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=compute_metrics)

smoke = build(epochs=1, out="./smoke")
smoke.train()

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1
1,0.136900,0.123338,0.793142


TrainOutput(global_step=210, training_loss=0.18184342781702678, metrics={'train_runtime': 65.5372, 'train_samples_per_second': 102.537, 'train_steps_per_second': 3.204, 'total_flos': 226530056895360.0, 'train_loss': 0.18184342781702678, 'epoch': 1.0})

In [11]:
trainer = build(epochs=8, out="./checkpoints/validation")
trainer.train()

Some weights of BertForTokenClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1
1,0.144600,0.134807,0.794380
2,0.083800,0.131366,0.792252
3,0.059600,0.140047,0.806765
4,0.037100,0.152182,0.802222
5,0.030700,0.167935,0.812172
6,0.023900,0.189569,0.806381
7,0.016900,0.185228,0.805339
8,0.020900,0.205553,0.807671


TrainOutput(global_step=1680, training_loss=0.05813526324040833, metrics={'train_runtime': 554.2285, 'train_samples_per_second': 97.0, 'train_steps_per_second': 3.031, 'total_flos': 1799713522672704.0, 'train_loss': 0.05813526324040833, 'epoch': 8.0})

In [12]:
preds, labels, _ = trainer.predict(tokenized[EVAL_SPLIT])
preds = np.argmax(preds, axis=2)
tp = [[ID2LABEL[a] for a, b in zip(pr, lb) if b != -100] for pr, lb in zip(preds, labels)]
tl = [[ID2LABEL[b] for a, b in zip(pr, lb) if b != -100] for pr, lb in zip(preds, labels)]
print(classification_report(tl, tp, digits=3))

trainer.save_model("./checkpoints/validation")
tokenizer.save_pretrained("./checkpoints/validation")

              precision    recall  f1-score   support

         EVT      0.371     0.408     0.388       130
         FNB      0.824     0.884     0.853        95
         IND      0.721     0.753     0.737       373
         PLC      0.807     0.834     0.820       523
         PPL      0.910     0.930     0.920       644

   micro avg      0.793     0.823     0.808      1765
   macro avg      0.726     0.762     0.744      1765
weighted avg      0.795     0.823     0.809      1765



('./checkpoints/validation/tokenizer_config.json',
 './checkpoints/validation/special_tokens_map.json',
 './checkpoints/validation/vocab.txt',
 './checkpoints/validation/added_tokens.json',
 './checkpoints/validation/tokenizer.json')

In [13]:
main_export(
    model_name_or_path="./checkpoints/validation",
    output="./onnx_fp32",
    task="token-classification",
    opset=18,  
    legacy=True,
)

m = ORTModelForTokenClassification.from_pretrained("./onnx_fp32")
tokenizer = BertTokenizerFast.from_pretrained("./checkpoints/validation")
tokenizer.save_pretrained("./onnx_fp32")

ORTQuantizer.from_pretrained("./onnx_fp32").quantize(
    save_dir="./onnx_int8",
    quantization_config=AutoQuantizationConfig.avx2(is_static=False, per_channel=False),
)
tokenizer.save_pretrained("./onnx_int8")

for d in ["./onnx_fp32", "./onnx_int8"]:
    mb = sum(f.stat().st_size for f in Path(d).glob("*.onnx")) / 1024**2
    print(f"{d}: {mb:.1f} MB")

m = ORTModelForTokenClassification.from_pretrained(
    "./checkpoints/validation",
    export=True
)

m.save_pretrained("./onnx_fp32"); tokenizer.save_pretrained("./onnx_fp32")

ORTQuantizer.from_pretrained("./onnx_fp32").quantize(
    save_dir="./onnx_int8",
    quantization_config=AutoQuantizationConfig.avx2(is_static=False, per_channel=False))
tokenizer.save_pretrained("./onnx_int8")

for d in ["./onnx_fp32", "./onnx_int8"]:
    mb = sum(f.stat().st_size for f in Path(d).glob("*.onnx")) / 1024**2
    print(f"{d}: {mb:.1f} MB")

./onnx_fp32: 472.7 MB
./onnx_int8: 118.9 MB
./onnx_fp32: 472.7 MB
./onnx_int8: 118.9 MB


In [14]:
for path in sorted(glob.glob("onnx_*/*.onnx")):
    m = onnx.load(path)
    types = Counter(i.data_type for i in m.graph.initializer)   
    print(f"\n{path}  {dict(types)}")
    big = sorted(m.graph.initializer, key=lambda i: len(i.raw_data), reverse=True)[:5]
    for i in big:
        print(f"   {i.name[:55]:<55} {len(i.raw_data)/1024**2:6.1f} MB  dtype={i.data_type}")


onnx_fp32/model.onnx  {1: 199}
   bert.embeddings.word_embeddings.weight                   146.5 MB  dtype=1
   onnx::MatMul_1635                                          9.0 MB  dtype=1
   onnx::MatMul_1636                                          9.0 MB  dtype=1
   onnx::MatMul_1648                                          9.0 MB  dtype=1
   onnx::MatMul_1649                                          9.0 MB  dtype=1

onnx_int8/model_quantized.onnx  {1: 199, 2: 152}
   bert.embeddings.word_embeddings.weight_quantized          36.6 MB  dtype=2
   onnx::MatMul_1635_quantized                                2.2 MB  dtype=2
   onnx::MatMul_1636_quantized                                2.2 MB  dtype=2
   onnx::MatMul_1648_quantized                                2.2 MB  dtype=2
   onnx::MatMul_1649_quantized                                2.2 MB  dtype=2


In [15]:
Path("onnx_int8_v2").mkdir(parents=True, exist_ok=True)

quant_pre_process(
    input_model_path="onnx_fp32/model.onnx",
    output_model_path="onnx_fp32/model_preprocessed.onnx",
    skip_symbolic_shape=True,
)

In [16]:
Path("onnx_int8_v3").mkdir(parents=True, exist_ok=True)

quant_pre_process(
    input_model_path="onnx_fp32/model.onnx",
    output_model_path="onnx_fp32/model_preprocessed.onnx",
    skip_symbolic_shape=True,
)

In [17]:
quantize_dynamic(
    "onnx_fp32/model_preprocessed.onnx", "onnx_int8_v2/model.onnx",
    weight_type=QuantType.QInt8,
    op_types_to_quantize=["MatMul"],
    extra_options={"MatMulConstBOnly": False}
)

In [18]:
quantize_dynamic(
    "onnx_fp32/model_preprocessed.onnx", "onnx_int8_v3/model.onnx",
    weight_type=QuantType.QInt8,
    op_types_to_quantize=["MatMul", "Gather"],
    extra_options={"MatMulConstBOnly": False},
)

In [19]:
for path in sorted(glob.glob("onnx_*/*.onnx")):
    m = onnx.load(path)
    types = Counter(i.data_type for i in m.graph.initializer)   
    print(f"\n{path}  {dict(types)}")
    big = sorted(m.graph.initializer, key=lambda i: len(i.raw_data), reverse=True)[:5]
    for i in big:
        print(f"   {i.name[:55]:<55} {len(i.raw_data)/1024**2:6.1f} MB  dtype={i.data_type}")


onnx_fp32/model.onnx  {1: 199}
   bert.embeddings.word_embeddings.weight                   146.5 MB  dtype=1
   onnx::MatMul_1635                                          9.0 MB  dtype=1
   onnx::MatMul_1636                                          9.0 MB  dtype=1
   onnx::MatMul_1648                                          9.0 MB  dtype=1
   onnx::MatMul_1649                                          9.0 MB  dtype=1

onnx_fp32/model_preprocessed.onnx  {1: 204, 7: 59}
   bert.embeddings.word_embeddings.weight                   146.5 MB  dtype=1
   onnx::MatMul_1635                                          9.0 MB  dtype=1
   onnx::MatMul_1636                                          9.0 MB  dtype=1
   onnx::MatMul_1648                                          9.0 MB  dtype=1
   onnx::MatMul_1649                                          9.0 MB  dtype=1

onnx_int8/model_quantized.onnx  {1: 199, 2: 152}
   bert.embeddings.word_embeddings.weight_quantized          36.6 MB  dtype=2
   onnx:

In [20]:
tok = BertTokenizerFast.from_pretrained("onnx_fp32")
text = "bu indomi goreng 1 dus sm gula 5kg, minyak 2 jrigen yg biasa"

for path in sorted(glob.glob("onnx_*/*.onnx")):
    sess = ort.InferenceSession(path)
    enc  = tok(text, return_tensors="np")
    feed = {i.name: enc[i.name].astype(np.int64) for i in sess.get_inputs()}
    for _ in range(5): sess.run(None, feed)
    t0 = time.perf_counter()
    for _ in range(50): sess.run(None, feed)
    print(f"{path:<40} {(time.perf_counter()-t0)/50*1000:6.1f} ms")

onnx_fp32/model.onnx                       47.2 ms
onnx_fp32/model_preprocessed.onnx          46.6 ms
onnx_int8/model_quantized.onnx             23.4 ms
onnx_int8_v2/model.onnx                    17.9 ms
onnx_int8_v3/model.onnx                    18.1 ms


In [21]:
def onnx_f1(path, dataset, batch=32):
    sess = ort.InferenceSession(path)
    names = [i.name for i in sess.get_inputs()]
    tp, tl = [], []
    for s in range(0, len(dataset), batch):
        chunk = dataset[s:s+batch]
        maxlen = max(len(x) for x in chunk["input_ids"])
        feed, labs = {}, []
        for n in names:
            feed[n] = np.array([x + [0]*(maxlen-len(x)) for x in chunk[n]], dtype=np.int64)
        labs = [x + [-100]*(maxlen-len(x)) for x in chunk["labels"]]
        logits = sess.run(None, feed)[0]
        preds = logits.argmax(-1)
        for p_row, l_row in zip(preds, labs):
            tp.append([ID2LABEL[p] for p, l in zip(p_row, l_row) if l != -100])
            tl.append([ID2LABEL[l] for p, l in zip(p_row, l_row) if l != -100])
    return f1_score(tl, tp), tl, tp

for path in sorted(glob.glob("onnx_*/*.onnx")):
    f1, _, _ = onnx_f1(path, tokenized[EVAL_SPLIT])
    print(f"{path:<40} F1 {f1:.4f}")

onnx_fp32/model.onnx                     F1 0.8077
onnx_fp32/model_preprocessed.onnx        F1 0.8077
onnx_int8/model_quantized.onnx           F1 0.8077
onnx_int8_v2/model.onnx                  F1 0.8106
onnx_int8_v3/model.onnx                  F1 0.8088
